In [ ]:
from time import sleep as delay
from project.el5600 import project
if "ic" in globals():
    ic.close()
ic = project(device="el5600", revision="a1", emulator="cp2112", logging=False, is_gui=False)

from phy.multimeter import keithley_dm6500, rohde_hmc8012, dmm
from phy.power_supply import keithley_2470, keysight_e36232a
from phy.scope import tektronix_dp_series_usb
from phy.eloader import sdl1030x
# from phy.relay_16ch import relay_box
# from phy.chamber_th3 import chamber

%matplotlib tk
from interface.docs.output_chart import plot
from interface.cui_colors import color
from interface.i2c_bridge.tca9548a import tca9548
from interface.docs.output_excel import excel_frame, style
from interface.cui_logger import logger as log

import numpy as np

chart = plot()

dm1 = keithley_dm6500()
dm2 = rohde_hmc8012()
ps = keysight_e36232a()
sm = keithley_2470()
ld = sdl1030x()
ds = tektronix_dp_series_usb()

# relay = relay_box(i2c_h=ic)
# tc = chamber(port=3)
# mux = tca9548(ic.i2c_h, 0x70)

# --------------------------------------------------
# list_voltage = list(np.arange(5, 8, 0.005))
# voltage  = [round(num, 3) for num in list_voltage]
# --------------------------------------------------

In [ ]:
def disable_all():
    
    sm.disable
    ps.disable
    ld.disable

In [ ]:
log.output_set_filename(f"[EL6200_A1, LGE sample] Basic function test_evb#2")

In [ ]:
log.output_csv(["VDD3V3 (V)", "I_VDD3V3 (uA)", "VIN_POK_STS", "SW_STS"])

In [ ]:
disable_all()

In [ ]:
# VDD3V3 : UVLO, IQ
list_item = list(np.arange(0.1, 5, 0.1))
for_item  = [round(num, 1) for num in list_item]

sm.cfg_all = 0.1, 0.01
sm.enable
delay(1)

count = 0

for set_vdd in for_item:
    
    sm.vset = set_vdd
    curr = round(sm.current*1e+6, 6)
    
    try:
        vin_pok_sts = ic.VIN_POK_STS
        sw_sts = ic.SW_STS
        count += 1
    except:
        vin_pok_sts = "NACK"
        sw_sts = "NACK"
    
    ret = [set_vdd, curr, vin_pok_sts, sw_sts]
    log.output_csv(ret)
    
    if count > 15:
        disable_all()
        break

In [ ]:
log.output_csv(["VIN (V)", "I_VIN (uA)", "VIN_POK_STS", "SW_STS"])

In [ ]:
# VIN : UVLO, IQ
list_item = list(np.arange(0.1, 30.01, 0.5))
for_item  = [round(num, 1) for num in list_item]

sm.cfg_all = 3.3, 0.01
sm.enable
delay(1)

ps.cfg_all = 0.1, 0.1
ps.enable

for set_vin in for_item:
    
    ps.vset = set_vin
    curr = round(dm1.current_10E_3 * 1e+3, 6)
    
    try:
        vin_pok_sts = ic.VIN_POK_STS
        sw_sts = ic.SW_STS
    except:
        vin_pok_sts = "NACK"
        sw_sts = "NACK"
    
    ret = [set_vin, curr, vin_pok_sts, sw_sts]
    log.output_csv(ret)
    print(ret)

disable_all()

In [ ]:
log.output_csv(["VIN (V)", "I_VIN (A)", "OC_FAULT", "SW_STS", "ILIM_TH_EX", "ILIM_TH"])

In [ ]:
# OC_FAULT
list_item = list(np.arange(2.5, 4, 0.02))
for_item  = [round(num, 2) for num in list_item]

dm2.current_10A

sm.cfg_all = 3.3, 0.01
sm.enable
delay(1)

ps.cfg_all = 20, 5
ps.enable
delay(1)

ld.iset = 1
ld.enable
delay(1)

# 3.25A
ic.ILIM_TH_EX = 0
ic.ILIM_TH = 0

count = 0

for set_load in for_item:
    
    ld.iset = set_load
    curr = round(dm2.current_10A, 6)
    oc_fault = round(dm1.voltage_10, 3)
    sw_sts = ic.SW_STS

    ret = [set_load, curr, oc_fault, sw_sts, 0, 0]
    print(ret)
    log.output_csv(ret)
    
    if sw_sts == 0:
        ld.disable
        count += 1
        
    if count > 10:
        ld.iset = 1
        break

ic.status
disable_all()

In [ ]:
log.output_csv(["VIN (V)", "V_INFO", "RATIO"])

In [ ]:
# OC_FAULT
list_item = list(np.arange(1, 30.01, 1))
for_item  = [round(num, 2) for num in list_item]

sm.cfg_all = 3.3, 0.01
sm.enable
delay(1)

ps.cfg_all = 1, 1
ps.enable
delay(1)

ic.CONV_VIN_EN = 1

for set_vin in for_item:
    
    ps.vset = set_vin
    v_info = round(dm1.voltage_10, 6)
    ratio = round(v_info / set_vin, 6)

    ret = [set_vin, v_info, ratio]
    print(ret)
    log.output_csv(ret)

disable_all()

In [ ]:
log.output_csv(["VOUT (V)", "V_INFO", "RATIO"])

In [ ]:
# OC_FAULT
disable_all()

list_item = list(np.arange(1, 30.01, 1))
for_item  = [round(num, 2) for num in list_item]

sm.cfg_all = 3.3, 0.01
sm.enable
delay(1)

ps.cfg_all = 1, 1
ps.enable
delay(1)

ic.CONV_VOUT_EN = 1

for set_vout in for_item:
    
    ps.vset = set_vout
    v_info = round(dm1.voltage_10, 6)
    ratio = round(v_info / set_vout, 6)

    ret = [set_vout, v_info, ratio]
    print(ret)
    log.output_csv(ret)

disable_all()

In [ ]:
ic.register_dump()